In [1]:
import pandas as pd

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [ ]:
df_2 = pd.read_excel(data_path('PlacementHistory.xlsx'))
df_2. columns.to_list

In [58]:
Total_Student_matches = df_2.groupby('Group Name')['Student Code'].nunique()
Total_agencies_matched = df_2.groupby('Group Name')['Agency Id'].nunique()
Total_Uagencies_matched = df_2['Agency Id'].nunique()

bulk_matches = df_2[df_2["Created By"] == "elijahnunez@gmail.com"].groupby('Group Name')['Student Code'].nunique()

pre_matches = df_2[
    (df_2["Created By"] != "elijahnunez@gmail.com") & 
    (df_2["Changed Date"] <= "2026-02-02") &
    (df_2["Changed Date"].notna()) 
].groupby('Group Name')['Student Code'].nunique()

manual_matches = df_2[
    (df_2["Created By"] != "elijahnunez@gmail.com") & 
    (df_2["Changed Date"] >= "2026-02-03") & 
    (df_2["Changed Date"] <= "2026-02-06")
].groupby('Group Name')['Student Code'].nunique()

summary = pd.DataFrame({
    'Unique Students Matched': Total_Student_matches,
    'Unique Agencies Matched': Total_agencies_matched,
    'Accepted Pre Matches':     pre_matches,
    'Accepted Bulk Matches':    bulk_matches,
    'Accepted Manual Matches':  manual_matches,
}).fillna(0).astype(int)

print(summary.to_string())
print(f"Total Unique Agencies Matched: {Total_Uagencies_matched}")


                                                     Unique Students Matched  Unique Agencies Matched  Accepted Pre Matches  Accepted Bulk Matches  Accepted Manual Matches
Group Name                                                                                                                                                                 
2026 Spring Forward - Community and Social Services                      218                       98                    61                     74                       69
2026 Spring Forward - Healthcare                                         228                       68                    42                    146                       54
2026 Spring Forward - Marketing and Communications                       244                       98                     7                    212                       14
2026 Spring Forward - STEM and Green                                     243                       64                    37                 

In [ ]:
df = pd.read_excel(data_path('SF26StudentOpportunityApplications.xlsx'))
df.head()

In [ ]:
placed_df = df[df['Opportunity Application Status'] == 'Placed']

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

# ── Prep ──────────────────────────────────────────────────────────────────────
placed_df['StudentPreference'] = pd.to_numeric(placed_df['StudentPreference'], errors='coerce').fillna(0).astype(int)
placed_df['Coordinator Preference Rank'] = pd.to_numeric(placed_df['Coordinator Preference Rank'], errors='coerce').fillna(0).astype(int)

hubs = placed_df['Opportunity Campaign Name'].dropna().unique()


# ══════════════════════════════════════════════════════════════
# 1. SMALL MULTIPLES — histogram per hub
# ══════════════════════════════════════════════════════════════
ncols = 2
nrows = int(np.ceil(len(hubs) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5), facecolor='#f5f5f5')
axes = axes.flatten()

bins = range(0, 54)
x = np.arange(0, 53)
width = 0.4




for i, hub in enumerate(sorted(hubs)):
    ax = axes[i]
    hub_df = placed_df[placed_df['Opportunity Campaign Name'] == hub]

    max_rank = max(
    hub_df['StudentPreference'].max(),
    hub_df['Coordinator Preference Rank'].max()
    )

    student_counts, _ = np.histogram(hub_df['StudentPreference'].dropna(), bins=bins)
    coord_counts, _   = np.histogram(hub_df['Coordinator Preference Rank'].dropna(), bins=bins)

    ax.bar(x - width/2, student_counts, width=width, color='#60a5fa', edgecolor='white', label='Student Pref')
    ax.bar(x + width/2, coord_counts,   width=width, color='#4ade80', edgecolor='white', label='Coordinator Rank')

    ax.set_title(hub, fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('Rank', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.tick_params(labelsize=10)
    ax.set_facecolor('white')
    ax.set_xlim(-1, max_rank + 1)
    for spine in ax.spines.values():
        spine.set_edgecolor('#e5e7eb')

axes[0].legend(fontsize=8)
for j in range(len(hubs), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Student vs Coordinator Preference Distribution by Hub',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
HISTOGRAMS_PATH = outputs_path('matching', 'preference_histograms.png')
os.makedirs(os.path.dirname(HISTOGRAMS_PATH), exist_ok=True)
plt.savefig(HISTOGRAMS_PATH, dpi=150, bbox_inches='tight', facecolor='#f5f5f5')
print(f"Saved: {HISTOGRAMS_PATH}")

# ══════════════════════════════════════════════════════════════
# 2. BUBBLE CHART — Student Rank vs Coordinator Rank
# ══════════════════════════════════════════════════════════════
pivot = placed_df.groupby(
    ['StudentPreference', 'Coordinator Preference Rank']
).size().reset_index(name='count')

# remove 0/NR if you want, or keep for unranked
pivot = pivot[pivot['count'] > 0]

fig2, ax = plt.subplots(figsize=(16, 14), facecolor='white')

scatter = ax.scatter(
    pivot['Coordinator Preference Rank'],
    pivot['StudentPreference'],
    s=pivot['count'] * 40,        # scale bubble size
    c=pivot['count'],
    cmap='RdYlGn',
    alpha=0.75,
    edgecolors='white',
    linewidths=0.8
)

# annotate count inside each bubble
for _, row in pivot.iterrows():
    if row['count'] > 1:
        ax.text(row['Coordinator Preference Rank'], row['StudentPreference'],
                str(int(row['count'])), ha='center', va='center',
                fontsize=7, fontweight='bold', color='#111827')

# diagonal — perfect match line
max_val = max(pivot['StudentPreference'].max(), pivot['Coordinator Preference Rank'].max())
ax.plot([0, max_val], [0, max_val], '--', color='#94a3b8', linewidth=1, label='Perfect match')

plt.colorbar(scatter, ax=ax, label='Number of Students', shrink=0.6)

ax.set_xlabel('Coordinator Preference Rank', fontsize=12, labelpad=10)
ax.set_ylabel('Student Preference', fontsize=12, labelpad=10)
ax.set_title('Student Preference vs Coordinator Preference Rank',
             fontsize=14, fontweight='bold', pad=16)
ax.legend(fontsize=9)
ax.set_facecolor('#f8fafc')
ax.grid(color='#e2e8f0', linewidth=0.6)
for spine in ax.spines.values():
    spine.set_edgecolor('#e2e8f0')

plt.tight_layout()
BUBBLE_PATH = outputs_path('matching', 'preference_bubble.png')
plt.savefig(BUBBLE_PATH, dpi=150, bbox_inches='tight')
print(f"Saved: {BUBBLE_PATH}")

In [ ]:

# Single combined chart across all hubs
bins = range(0, 54)
x = np.arange(0, 53)
width = 0.4

max_rank = max(
    placed_df['StudentPreference'].max(),
    placed_df['Coordinator Preference Rank'].max()
)

student_counts, _ = np.histogram(placed_df['StudentPreference'].dropna(), bins=bins)
coord_counts, _   = np.histogram(placed_df['Coordinator Preference Rank'].dropna(), bins=bins)

fig, ax = plt.subplots(figsize=(18, 5), facecolor='#f5f5f5')

ax.bar(x - width/2, student_counts, width=width, color='#60a5fa', edgecolor='white', label='Student Pref')
ax.bar(x + width/2, coord_counts,   width=width, color='#4ade80', edgecolor='white', label='Coordinator Rank')

ax.set_title('Student vs Coordinator Preference Distribution (All Hubs)', fontsize=12, fontweight='bold', pad=8)
ax.set_xlabel('Rank', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_xlim(-1, max_rank + 1)
ax.set_facecolor('white')
ax.legend(fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#e5e7eb')

plt.tight_layout()
COMBINED_PATH = outputs_path('matching', 'preference_histogram_combined.png')
plt.savefig(COMBINED_PATH, dpi=150, bbox_inches='tight', facecolor='#f5f5f5')
plt.show()
print(f"Saved: {COMBINED_PATH}")


In [ ]:
import folium

map_df = df_2.dropna(subset=['Postal Address Latitude', 'Postal Address Longitude']).copy()

# Aggregate by location + group name
location_group = map_df.groupby(
    ['Postal Address Latitude', 'Postal Address Longitude', 'Group Name']
)['Student Code'].nunique().reset_index()
location_group.columns = ['lat', 'lon', 'Group Name', 'student_count']

# Assign a color to each unique Group Name (hub)
groups = location_group['Group Name'].unique()
colors = [
    '#4ade80', '#60a5fa', '#f97316', '#a78bfa', '#f472b6',
    '#34d399', '#fbbf24', '#fb7185', '#38bdf8', '#e879f9'
]
color_map = {group: colors[i % len(colors)] for i, group in enumerate(groups)}

m = folium.Map(
    location=[map_df['Postal Address Latitude'].mean(),
              map_df['Postal Address Longitude'].mean()],
    zoom_start=11,
    tiles='CartoDB positron'
)

for _, row in location_group.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['student_count'] * 2,  # scale size by student count
        color=color_map[row['Group Name']],
        fill=True,
        fill_color=color_map[row['Group Name']],
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>{row['Group Name']}</b><br>Students: {row['student_count']}",
            max_width=200
        )
    ).add_to(m)

# Add legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px; border-radius:8px;
     font-family:sans-serif; font-size:12px; box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
  <b>Hubs</b><br>
"""
for group, color in color_map.items():
    legend_html += f'<span style="background:{color};width:12px;height:12px;display:inline-block;border-radius:50%;margin-right:6px"></span>{group}<br>'
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

m.save('student_map.html')
print(f"Total locations plotted: {len(location_group)}")

Total locations plotted: 277


In [57]:


print(summary.to_string())



                                                     Unique Students Matched  Unique Agencies Matched  Accepted Pre Matches  Accepted Bulk Matches  Accepted Manual Matches
Group Name                                                                                                                                                                 
2026 Spring Forward - Community and Social Services                      218                       98                    61                     74                       69
2026 Spring Forward - Healthcare                                         228                       68                    42                    146                       54
2026 Spring Forward - Marketing and Communications                       244                       98                     7                    212                       14
2026 Spring Forward - STEM and Green                                     243                       64                    37                 

In [60]:
# Merge df with df_2 to get Agency Name, then count applications by hub + agency
agency_name_map = df_2[['Agency Id', 'Agency Name']].drop_duplicates()

apps_by_hub_agency = (
    df
    .merge(agency_name_map, on='Agency Id', how='left')
    .groupby(['Opportunity Campaign Name', 'Agency Name'])
    .size()
    .reset_index(name='Applications')
    .sort_values(['Opportunity Campaign Name', 'Applications'], ascending=[True, False])
)

# Show top 10 agencies per hub
top_agencies = apps_by_hub_agency.groupby('Opportunity Campaign Name').head(10)

for hub, group in top_agencies.groupby('Opportunity Campaign Name'):
    print(f"\n{'='*60}")
    print(f"  {hub}")
    print(f"{'='*60}")
    print(group[['Agency Name', 'Applications']].to_string(index=False))


  2026 Spring Forward - Community and Social Services
                                                                  Agency Name  Applications
                                                Center for Justice Innovation           126
                                                      Public Health Solutions           110
                                                       Stand Out College Prep            87
                                              Bridge Philanthropic Consulting            81
                                                 YWCA of the City of New York            79
                                  New York City Department of Social Services            69
                                                    Catskill Animal Sanctuary            56
                                                                Girl Vow Inc.            56
                                                       9/11 Memorial & Museum            53
CUNY Office of Career and

In [61]:

from tabulate import tabulate

# Merge df with df_2 to get Agency Name, then count applications by hub + agency
agency_name_map = df_2[['Agency Id', 'Agency Name']].drop_duplicates()

apps_by_hub_agency = (
    df
    .merge(agency_name_map, on='Agency Id', how='left')
    .groupby(['Opportunity Campaign Name', 'Agency Name'])
    .size()
    .reset_index(name='Applications')
    .sort_values(['Opportunity Campaign Name', 'Applications'], ascending=[True, False])
)

# Top 10 agencies per hub
top_agencies = (
    apps_by_hub_agency
    .groupby('Opportunity Campaign Name')
    .head(10)
    .rename(columns={'Opportunity Campaign Name': 'Hub'})
    .reset_index(drop=True)
)
top_agencies.index += 1

print(tabulate(top_agencies, headers='keys', tablefmt='simple', showindex=False))

Hub                                                  Agency Name                                                                      Applications
---------------------------------------------------  -----------------------------------------------------------------------------  --------------
2026 Spring Forward - Community and Social Services  Center for Justice Innovation                                                             126
2026 Spring Forward - Community and Social Services  Public Health Solutions                                                                   110
2026 Spring Forward - Community and Social Services  Stand Out College Prep                                                                     87
2026 Spring Forward - Community and Social Services  Bridge Philanthropic Consulting                                                            81
2026 Spring Forward - Community and Social Services  YWCA of the City of New York                                     

In [ ]:

# Employer ranking completion by hub
# An employer "completed ranking" if they assigned at least one non-null, non-zero Coordinator Preference Rank

ranked_mask = df['Coordinator Preference Rank'].notna() & (df['Coordinator Preference Rank'] != 0)

total_employers = (
    df.groupby('Opportunity Campaign Name')['Agency Id']
    .nunique()
    .rename('Total Employers')
)

ranked_employers = (
    df[ranked_mask]
    .groupby('Opportunity Campaign Name')['Agency Id']
    .nunique()
    .rename('Employers Who Ranked')
)

employer_ranking_summary = pd.concat([total_employers, ranked_employers], axis=1).fillna(0).astype(int)
employer_ranking_summary['% Completed Ranking'] = (
    employer_ranking_summary['Employers Who Ranked'] / employer_ranking_summary['Total Employers'] * 100
).round(1)

employer_ranking_summary.index = employer_ranking_summary.index.str.replace('2026 Spring Forward - ', '', regex=False)
employer_ranking_summary.index.name = 'Hub'

# Add total row
total_row = pd.DataFrame({
    'Total Employers': [df['Agency Id'].nunique()],
    'Employers Who Ranked': [df[ranked_mask]['Agency Id'].nunique()],
    '% Completed Ranking': [round(df[ranked_mask]['Agency Id'].nunique() / df['Agency Id'].nunique() * 100, 1)]
}, index=pd.Index(['TOTAL'], name='Hub'))

employer_ranking_summary = pd.concat([employer_ranking_summary, total_row])

print(employer_ranking_summary.to_string())


In [ ]:

# Average max rankings employers give, by hub
ranked = df[df['Coordinator Preference Rank'].notna() & (df['Coordinator Preference Rank'] != 0)].copy()

# Max rank given per employer per hub
max_rank_per_employer = (
    ranked.groupby(['Opportunity Campaign Name', 'Agency Id'])['Coordinator Preference Rank']
    .max()
    .reset_index(name='Max Rank Given')
)

avg_max_rank = (
    max_rank_per_employer.groupby('Opportunity Campaign Name')['Max Rank Given']
    .agg(Avg_Max_Rank='mean', Median_Max_Rank='median', Std_Max_Rank='std', Min_Max_Rank='min', Max_Max_Rank='max')
    .round(1)
)

avg_max_rank.index = avg_max_rank.index.str.replace('2026 Spring Forward - ', '', regex=False)
avg_max_rank.index.name = 'Hub'

# Overall
overall = max_rank_per_employer['Max Rank Given'].agg(['mean', 'median', 'std', 'min', 'max']).round(1)
overall_row = pd.DataFrame({
    'Avg_Max_Rank': [overall['mean']],
    'Median_Max_Rank': [overall['median']],
    'Std_Max_Rank': [overall['std']],
    'Min_Max_Rank': [overall['min']],
    'Max_Max_Rank': [overall['max']]
}, index=pd.Index(['TOTAL'], name='Hub'))

print(pd.concat([avg_max_rank, overall_row]).to_string())
